In [ ]:
# ─────────────────────────────────────────────
# SECTION 1 — Install & Imports
# ─────────────────────────────────────────────
!pip install shap -q

import shap
import re, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('All imports successful.')
print(f'SHAP version: {shap.__version__}')


In [ ]:
# ─────────────────────────────────────────────
# SECTION 2 — Load & Filter Product Dataset
# ─────────────────────────────────────────────
products = pd.read_csv('skincare_products.csv')

print('Shape:', products.shape)
print('Columns:', products.columns.tolist())
display(products.head(3))

ing_col = next((c for c in products.columns if 'ingred' in c.lower()), None)
if ing_col is None:
    raise ValueError(f'No ingredients column found. Columns: {products.columns.tolist()}')
print(f'\nIngredient column detected: "{ing_col}"')

BRAND_CANDIDATES = ['brand', 'Brand', 'brand_name', 'BrandName', 'manufacturer']
TYPE_CANDIDATES  = ['product_type', 'type', 'Label', 'category', 'Category',
                    'product_category', 'skin_type']

brand_col = next((c for c in BRAND_CANDIDATES if c in products.columns), None)
type_col  = next((c for c in TYPE_CANDIDATES  if c in products.columns), None)

print(f'Brand column : {brand_col or "NOT FOUND — will be imputed from name"}')
print(f'Type column  : {type_col  or "NOT FOUND — will be imputed from ingredients"}')

SKINCARE_TYPES = {
    'moisturizer', 'moisturiser', 'serum', 'cleanser', 'face wash',
    'toner', 'gel', 'cream', 'lotion', 'sunscreen', 'spf', 'face mask',
    'spot treatment', 'treatment', 'exfoliant', 'essence', 'oil', 'mist',
    'eye cream', 'balm', 'primer',
}

def is_skincare(row):
    """Return True only if the product is a face/skincare product."""
    if type_col and pd.notna(row.get(type_col)):
        label = str(row[type_col]).lower()
        # Exclude if label explicitly contains body / hair / bath keywords
        if any(bad in label for bad in ['body', 'hair', 'shampoo', 'conditioner',
                                         'bath', 'hand', 'foot', 'lip', 'nail']):
            return False
        # Include if label matches skincare type
        if any(good in label for good in SKINCARE_TYPES):
            return True
    # Fall back: if no type column, include (filter by ingredients alone)
    return True

ACNE_INGREDIENTS = [
    'salicylic acid', 'benzoyl peroxide', 'niacinamide', 'azelaic acid',
    'zinc pca', 'zinc gluconate', 'retinol', 'retinyl palmitate', 'adapalene',
    'glycolic acid', 'lactic acid', 'mandelic acid', 'sulfur', 'tea tree',
    'witch hazel', 'hyaluronic acid', 'caffeine', 'vitamin c', 'ascorbic acid',
    'kojic acid', 'alpha arbutin', 'tranexamic acid',
]

ing_mask  = products[ing_col].apply(
    lambda x: any(ing in str(x).lower() for ing in ACNE_INGREDIENTS)
    if pd.notna(x) else False
)
type_mask = products.apply(is_skincare, axis=1)

acne_products = products[ing_mask & type_mask].copy().reset_index(drop=True)

print(f'\nAll products           : {len(products)}')
print(f'Has acne ingredient    : {ing_mask.sum()}')
print(f'Is skincare type       : {type_mask.sum()}')
print(f'After both filters     : {len(acne_products)}')

if 'price' in acne_products.columns:
    acne_products['price_clean'] = pd.to_numeric(
        acne_products['price'].astype(str)
            .str.replace(r'[£$€,]', '', regex=True).str.strip(),
        errors='coerce'
    )

if brand_col is None:
    name_col_guess = next((c for c in ['product_name','name','Name','product name']
                           if c in acne_products.columns), acne_products.columns[0])
    acne_products['brand'] = acne_products[name_col_guess].astype(str).apply(
        lambda n: n.split()[0] if n and n.split() else 'Unknown'
    )
    brand_col = 'brand'
    print('\nℹ Brand column imputed from product name (first word).')

acne_products[brand_col] = acne_products[brand_col].fillna('Unknown')

print('\nTop brands in filtered dataset:')
print(acne_products[brand_col].value_counts().head(10))


In [ ]:
# ─────────────────────────────────────────────
# SECTION 3 — Parse Ingredient Lists
# ─────────────────────────────────────────────
def parse_ingredients(raw):
    if pd.isna(raw): return []
    parts = re.split(r'[,;]', str(raw))
    cleaned = []
    for p in parts:
        p = re.sub(r'\(.*?\)', '', p).strip().lower()
        p = re.sub(r'[*./\\]', '', p).strip()
        if p and len(p) > 1:
            cleaned.append(p)
    return cleaned

def get_acne_ings_present(ing_list):
    return [a for a in ACNE_INGREDIENTS
            if any(a in p for p in ing_list)]

acne_products['ingredients_list']      = acne_products[ing_col].apply(parse_ingredients)
acne_products['acne_ings_present']     = acne_products['ingredients_list'].apply(get_acne_ings_present)
acne_products['acne_ingredient_count'] = acne_products['acne_ings_present'].apply(len)

acne_products['acne_ings_str'] = acne_products['acne_ings_present'].apply(
    lambda x: '\x00'.join(x)
)

print('Parsing complete.')
print(acne_products[['acne_ings_str', 'acne_ingredient_count']].head(5))


In [ ]:
# ─────────────────────────────────────────────
# SECTION 4 — Auto-Tag Acne Types
# ─────────────────────────────────────────────
ACNE_TYPE_RULES = {
    'Comedonal'   : ['salicylic acid', 'zinc pca', 'glycolic acid', 'lactic acid', 'mandelic acid'],
    'Inflammatory': ['benzoyl peroxide', 'niacinamide', 'azelaic acid', 'tea tree', 'zinc gluconate'],
    'Cystic'      : ['benzoyl peroxide', 'azelaic acid', 'retinol', 'adapalene'],
    'Fungal'      : ['zinc pca', 'zinc gluconate', 'azelaic acid', 'sulfur'],
    'Pigmentation': ['niacinamide', 'vitamin c', 'ascorbic acid', 'kojic acid', 'alpha arbutin', 'tranexamic acid'],
    'Dark Circles': ['caffeine', 'vitamin c', 'hyaluronic acid'],
}

def assign_acne_types(ings_present):
    matched = [t for t, rule_ings in ACNE_TYPE_RULES.items()
               if any(i in ings_present for i in rule_ings)]
    return matched if matched else ['General']

acne_products['acne_types']     = acne_products['acne_ings_present'].apply(assign_acne_types)
acne_products['acne_types_str'] = acne_products['acne_types'].apply(lambda x: ', '.join(x))

print('Acne type tagging complete.')
from collections import Counter
all_types = [t for types in acne_products['acne_types'] for t in types]
for t, c in sorted(Counter(all_types).items(), key=lambda x: -x[1]):
    print(f'  {t:<20}: {c} products')


In [ ]:
# ─────────────────────────────────────────────
# SECTION 5 — Improved TF-IDF + Recommendation Model
# ─────────────────────────────────────────────

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

name_col_guess = next(
    (c for c in ['product_name','name','Name','product name','title'] if c in acne_products.columns),
    acne_products.columns[0]
)
print(f"Product name column: {name_col_guess!r}")

def build_rich_text(row):
    """
    Combine all available text fields for richer TF-IDF vocabulary.
    More text = more features = better discrimination.
    """
    parts = []

    # Product title (repeated 2x to upweight it)
    name = str(row.get(name_col_guess, ''))
    parts.extend([name, name])

    # Product type (repeated 2x)
    if type_col and pd.notna(row.get(type_col)):
        ptype = str(row[type_col])
        parts.extend([ptype, ptype])

    # Brand
    if brand_col and pd.notna(row.get(brand_col)):
        parts.append(str(row[brand_col]))

    # Acne-active ingredients (most important)
    ings_active = row.get('acne_ings_present', [])
    if isinstance(ings_active, list):
        parts.append(' '.join(ings_active))

    all_ings = row.get('ingredients_list', [])
    if isinstance(all_ings, list):
        parts.append(' '.join(all_ings[:30]))

    acne_types_list = row.get('acne_types', [])
    if isinstance(acne_types_list, list):
        parts.append(' '.join(acne_types_list))

    return ' '.join(filter(None, parts))


acne_products['rich_text'] = acne_products.apply(build_rich_text, axis=1)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_features=2000,     
    sublinear_tf=True,
    analyzer='word',
    token_pattern=r"(?u)\b[a-z][a-z]+\b",
    strip_accents='unicode',
)

product_matrix = tfidf.fit_transform(acne_products['rich_text'])
feature_names  = tfidf.get_feature_names_out()

print(f"\nRich TF-IDF matrix shape : {product_matrix.shape}")
print(f"Features (vocabulary size): {len(feature_names)}")
print(f"Improvement from original : {len(feature_names)} vs 14 features before")
print(f"Sample features           : {feature_names[:30].tolist()}")

acne_feats = [f for f in feature_names if any(
    ai.replace(' ','') in f.replace(' ','')
    for ai in ['salicylic', 'niacinamide', 'benzoyl', 'retinol', 'azelaic']
)]
print(f"\nAcne-related features found: {len(acne_feats)}")
print(f"  Sample: {acne_feats[:10]}")


ACNE_FAMILY = {
    'Comedonal'   : ['Comedonal', 'Blackhead', 'Whitehead'],
    'Inflammatory': ['Inflammatory', 'Pustule', 'Papule'],
    'Cystic'      : ['Cystic', 'Nodular'],
    'Fungal'      : ['Fungal'],
    'Pigmentation': ['Pigmentation', 'Dark Circles'],
}
UNIVERSAL_INGS = ['niacinamide', 'hyaluronic acid', 'ceramide', 'centella', 'panthenol']


def parse_formula(formula_str):
    """Parse 'Zinc PCA + Salicylic Acid(2%)' → ['zinc pca', 'salicylic acid']"""
    return [re.sub(r"\(.*?\)", "", p).strip().lower() for p in formula_str.split("+")]


def build_query_text(rec_ingredients):
    """Build rich query text from recommended ingredients list."""
    return " ".join(rec_ingredients)


def normalize_scores(scores):
    """Min-max normalize scores to [0, 1] range. FIX 7: prevents 100% overconfidence."""
    mn, mx = scores.min(), scores.max()
    if mx - mn < 1e-6:
        return np.ones_like(scores) * 0.5
    return (scores - mn) / (mx - mn)


def diversity_rerank(pool_df, top_n, penalty=0.25):
    """
    FIX 9: Penalize products that are too similar to already-selected products.
    This prevents recommending Effaclar H / CeraVe Lotion every single time.

    Algorithm:
    1. Select top product
    2. For remaining products, check brand duplication and ingredient similarity
    3. Apply penalty if same brand OR if >70% ingredient overlap with selected
    4. Repeat until top_n selected
    """
    selected = []
    selected_brands = set()
    selected_ing_sets = []
    remaining = pool_df.copy().reset_index(drop=True)

    while len(selected) < top_n and len(remaining) > 0:
        scores = remaining['hybrid_score'].copy()

        for i, row in remaining.iterrows():
            brand = str(row.get(brand_col, '')).lower().strip() if brand_col else ''
            prod_ings = set(str(i).lower() for i in row.get('acne_ings_present', []))

            if brand and brand in selected_brands and brand not in ('unknown', ''):
                scores[i] *= (1 - penalty)

            for sel_ings in selected_ing_sets:
                if sel_ings:
                    overlap = len(prod_ings & sel_ings) / max(len(prod_ings | sel_ings), 1)
                    if overlap > 0.7:
                        scores[i] *= (1 - penalty * 0.5)

        best_idx = scores.idxmax()
        best_row = remaining.loc[best_idx]
        selected.append(best_row)

        sel_brand = str(best_row.get(brand_col, '')).lower().strip() if brand_col else ''
        if sel_brand and sel_brand not in ('unknown', ''):
            selected_brands.add(sel_brand)
        sel_ings = set(str(i).lower() for i in best_row.get('acne_ings_present', []))
        selected_ing_sets.append(sel_ings)

        remaining = remaining.drop(index=best_idx).reset_index(drop=True)

    return pd.DataFrame(selected).reset_index(drop=True)


def find_matching_products(formula_str, acne_type=None, top_n=3,
                           min_shared_ings=1, apply_diversity=True):
    """
    FIX 10: Three-tier fallback matching system.

    Tier 1: Products tagged with exact acne type + ingredient overlap
    Tier 2: Products from same acne family if Tier 1 has < top_n results
    Tier 3: Universal safe products (ceramide/niacinamide/hyaluronic) as last resort

    Also applies:
    - FIX 6: uses rich_text TF-IDF (not 14-feature acne_ings_str)
    - FIX 7: normalized similarity scores
    - FIX 9: diversity reranking
    """
    rec_ingredients = parse_formula(formula_str)
    query_text      = build_query_text(rec_ingredients)
    query_vec       = tfidf.transform([query_text])
    query_tokens    = set(query_text.lower().split())

    # Cosine similarity on rich TF-IDF
    raw_sims = cosine_similarity(query_vec, product_matrix).flatten()

    def shared_count(row):
        prod_ings_str = str(row.get('acne_ings_str', '')).lower()
        return sum(1 for ri in rec_ingredients if ri in prod_ings_str)

    pool = acne_products.copy()
    pool['raw_sim']         = raw_sims
    pool['shared_ings']     = pool.apply(shared_count, axis=1)

    pool['tfidf_score']     = normalize_scores(raw_sims)

    def overlap_ratio(row):
        ings = row.get('acne_ings_present', [])
        if not ings or not rec_ingredients: return 0.0
        ing_str = " ".join(str(i).lower() for i in ings)
        matches = sum(1 for ri in rec_ingredients if ri in ing_str)
        return matches / max(len(rec_ingredients), 1)

    pool['overlap_score'] = pool.apply(overlap_ratio, axis=1)

    if acne_type:
        t1_mask = pool['acne_types'].apply(lambda t: acne_type in t)
        t1_pool = pool[t1_mask & (pool['shared_ings'] >= min_shared_ings)]

        if len(t1_pool) >= top_n:
            t1_pool = t1_pool.copy()
            t1_pool['hybrid_score'] = (
                0.15 * t1_pool['tfidf_score'] +  
                0.45 * 1.0 +                     
                0.30 * t1_pool['overlap_score'] +
                0.10 * 0.5                       
            )
            if apply_diversity:
                result = diversity_rerank(t1_pool, top_n)
            else:
                result = t1_pool.nlargest(top_n, 'hybrid_score').reset_index(drop=True)
            print(f"  Tier 1 match: {len(result)} products (exact acne type: {acne_type})")
            return result, query_vec, rec_ingredients

    if acne_type:
        family_types = []
        for fam, members in ACNE_FAMILY.items():
            if acne_type in members or fam == acne_type:
                family_types = members
                break

        if family_types:
            t2_mask = pool['acne_types'].apply(
                lambda t: any(ft in t for ft in family_types)
            )
            t2_pool = pool[t2_mask & (pool['shared_ings'] >= 1)].copy()

            if len(t2_pool) >= top_n:
                acne_match_sc = t2_pool['acne_types'].apply(
                    lambda t: 0.7 if any(ft in t for ft in family_types) else 0.0
                )
                t2_pool['hybrid_score'] = (
                    0.15 * t2_pool['tfidf_score'] +
                    0.45 * acne_match_sc +
                    0.30 * t2_pool['overlap_score'] +
                    0.10 * 0.5
                )
                if apply_diversity:
                    result = diversity_rerank(t2_pool, top_n)
                else:
                    result = t2_pool.nlargest(top_n, 'hybrid_score').reset_index(drop=True)
                print(f"  Tier 2 match: {len(result)} products (acne family: {family_types})")
                return result, query_vec, rec_ingredients

    print(f"   Tier 3 fallback: using universal safe products")
    def is_universal(row):
        ings = [str(i).lower() for i in row.get('acne_ings_present', [])]
        return any(ui in " ".join(ings) for ui in UNIVERSAL_INGS)

    t3_pool = pool[pool.apply(is_universal, axis=1)].copy()
    if t3_pool.empty:
        t3_pool = pool.copy()

    t3_pool['hybrid_score'] = (
        0.15 * t3_pool['tfidf_score'] +
        0.00 +  # no acne type match known
        0.75 * t3_pool['overlap_score'] +
        0.10 * 0.5
    )
    if apply_diversity:
        result = diversity_rerank(t3_pool, top_n)
    else:
        result = t3_pool.nlargest(top_n, 'hybrid_score').reset_index(drop=True)
    print(f"  Tier 3 match: {len(result)} products (universal safe)")
    return result, query_vec, rec_ingredients


print("\nImproved TF-IDF recommendation model ready.")
print(f"   Vocabulary: {len(feature_names)} features (was 14 before)")
print("   Tiers: Exact match → Family match → Universal fallback")
print("   Diversity reranking: enabled")
print("   Score normalization: enabled")

print("\n[Sanity check] Testing Comedonal query...")
test_r, _, test_ings = find_matching_products(
    "Salicylic Acid + Niacinamide + Zinc PCA",
    acne_type="Comedonal", top_n=3
)
tfidf_scores = test_r["tfidf_score"].values if "tfidf_score" in test_r else [0]
print(f"  TF-IDF scores: {[f'{s:.3f}' for s in tfidf_scores]}")
print("  If scores are NOT all 0.000, the rich TF-IDF fix worked!")


In [ ]:
# ─────────────────────────────────────────────
# SECTION 5B — Ingredient Conflict Rules (CLINICAL LOGIC)
# ─────────────────────────────────────────────

INGREDIENT_CONFLICTS = [
    {
        "name"       : "Benzoyl Peroxide + Retinol",
        "ingredients": ["benzoyl peroxide"],
        "conflicts"  : ["retinol", "retinyl palmitate", "tretinoin", "adapalene"],
        "reason"     : "BP oxidizes and deactivates retinoids. Never use in same step. "
                       "Use BP in AM, retinoid PM only.",
        "severity"   : "high",
    },
    {
        "name"       : "Benzoyl Peroxide + Vitamin C",
        "ingredients": ["benzoyl peroxide"],
        "conflicts"  : ["vitamin c", "ascorbic acid", "l-ascorbic acid"],
        "reason"     : "BP oxidizes Vitamin C, rendering both ineffective. "
                       "Use BP in PM, Vitamin C in AM.",
        "severity"   : "high",
    },
    {
        "name"       : "AHA/BHA + Retinol (same time)",
        "ingredients": ["glycolic acid", "lactic acid", "mandelic acid", "salicylic acid"],
        "conflicts"  : ["retinol", "tretinoin", "adapalene"],
        "reason"     : "Both are exfoliating. Combined use causes over-exfoliation, "
                       "barrier disruption, severe irritation. Alternate nights or "
                       "use one AM and one PM.",
        "severity"   : "high",
    },
    {
        "name"       : "Niacinamide + High-% Vitamin C",
        "ingredients": ["niacinamide"],
        "conflicts"  : ["ascorbic acid", "l-ascorbic acid"],
        "reason"     : "At high concentrations (>10% Vit C), these can form niacin "
                       "(causes skin flushing). Low % Vit C (<5%) is generally safe "
                       "with niacinamide. Modern research suggests this is overstated "
                       "at typical product concentrations, but worth noting.",
        "severity"   : "low",
    },
    {
        "name"       : "Multiple Acids Simultaneously",
        "ingredients": ["glycolic acid", "lactic acid"],
        "conflicts"  : ["salicylic acid", "mandelic acid", "azelaic acid"],
        "reason"     : "Combining multiple chemical exfoliants risks over-exfoliation "
                       "and pH conflict. Choose one exfoliant at a time.",
        "severity"   : "medium",
    },
]

SKIN_SENSITIVITY_RESTRICTIONS = {
    "sensitive": {
        "avoid"  : ["benzoyl peroxide", "fragrance", "parfum", "alcohol denat",
                    "glycolic acid", "salicylic acid"],
        "prefer" : ["niacinamide", "ceramide", "centella", "panthenol", "hyaluronic acid"],
        "reason" : "Sensitive skin has compromised barrier; strong actives cause flares.",
    },
    "dry": {
        "avoid"  : ["salicylic acid", "glycolic acid", "lactic acid", "alcohol denat"],
        "prefer" : ["hyaluronic acid", "ceramide", "squalane", "glycerin"],
        "reason" : "Dry skin is already barrier-compromised; exfoliating acids worsen dryness.",
    },
    "oily": {
        "avoid"  : ["heavy oils", "shea butter", "coconut oil"],
        "prefer" : ["salicylic acid", "niacinamide", "zinc pca"],
        "reason" : "Oily skin benefits from sebum-regulating actives and lightweight textures.",
    },
}


def check_ingredient_conflicts(formula_str, skin_sensitivity="normal"):
    """
    Check a formula string for ingredient conflicts.
    Returns list of (conflict_name, reason, severity) tuples.
    """
    formula_ings  = parse_formula(formula_str)
    conflicts_found = []

    for rule in INGREDIENT_CONFLICTS:
        has_primary = any(
            any(pi in ing for ing in formula_ings)
            for pi in rule["ingredients"]
        )
        if not has_primary:
            continue

        has_conflict = any(
            any(ci in ing or ing in ci for ing in formula_ings)
            for ci in rule["conflicts"]
        )
        if has_conflict:
            conflicts_found.append((rule["name"], rule["reason"], rule["severity"]))

    if skin_sensitivity.lower() in SKIN_SENSITIVITY_RESTRICTIONS:
        restr = SKIN_SENSITIVITY_RESTRICTIONS[skin_sensitivity.lower()]
        for avoid_ing in restr["avoid"]:
            if any(avoid_ing in ing for ing in formula_ings):
                conflicts_found.append((
                    f"Sensitivity Alert: {avoid_ing}",
                    f"{restr['reason']} Avoid {avoid_ing} for {skin_sensitivity} skin.",
                    "medium"
                ))

    return conflicts_found


def check_product_conflicts(product_ings_list_a, product_ings_list_b):
    """
    Check if two products should NOT be used together.
    Returns list of conflict descriptions.
    """
    combined_formula = " + ".join(
        list(product_ings_list_a[:5]) + list(product_ings_list_b[:5])
    )
    return check_ingredient_conflicts(combined_formula)


def format_conflict_warnings(conflicts):
    """Format conflicts into readable warning text."""
    if not conflicts:
        return "✓ No ingredient conflicts detected."
    lines = ["⚠ INGREDIENT CONFLICT WARNINGS:"]
    for name, reason, sev in conflicts:
        icon = "🔴" if sev == "high" else ("🟡" if sev == "medium" else "🟢")
        lines.append(f"  {icon} [{sev.upper()}] {name}")
        lines.append(f"     → {reason}")
    return "\n".join(lines)


print("Testing conflict detection:")
test_cases = [
    ("Benzoyl Peroxide + Retinol + Niacinamide", "normal"),
    ("Salicylic Acid + Glycolic Acid + Retinol", "normal"),
    ("Benzoyl Peroxide + Vitamin C + Zinc PCA", "sensitive"),
    ("Niacinamide + Ceramide + Hyaluronic Acid", "sensitive"),  # should be clean
]
for formula, skin_type in test_cases:
    conflicts = check_ingredient_conflicts(formula, skin_type)
    print(f"\n  Formula: {formula} [{skin_type} skin]")
    print(f"  {format_conflict_warnings(conflicts)}")

print("\n Ingredient conflict engine ready.")


In [ ]:
# ─────────────────────────────────────────────
# SECTION 6A — Similarity Model for SHAP
# ─────────────────────────────────────────────
X = product_matrix.toarray()

def train_similarity_model(formula: str):
    query_vec = tfidf.transform(['\x00'.join(parse_formula(formula))])
    sims      = cosine_similarity(query_vec, product_matrix).flatten()

    if sims.std() < 1e-6:
        print('⚠ Warning: similarity scores have no variance — SHAP will be uninformative')

    model = Ridge(alpha=1.0)
    model.fit(X, sims)
    print('Similarity model trained')
    return model, sims


In [ ]:
# ─────────────────────────────────────────────
# SECTION 6B — SHAP LinearExplainer
# ─────────────────────────────────────────────

def create_explainer(model):
    background = shap.maskers.Independent(X, max_samples=100)
    return shap.LinearExplainer(model, background)


def get_shap_explanation(product_index, shap_values, top_n=5):
    """
    Return top-n SHAP values for a product, restricted to its
    own ingredients (FIX 5: attribution tied to actual ingredient list).
    """
    fnames = tfidf.get_feature_names_out()
    sv     = shap_values[product_index]           # shape: (n_features,)

    # FIX 5: only keep features that are actually present in this product
    product_vector   = X[product_index]
    nonzero_features = fnames[product_vector > 0]

    shap_nonzero = {f: dict(zip(fnames, sv))[f] for f in nonzero_features}

    sorted_shap = dict(
        sorted(shap_nonzero.items(), key=lambda x: abs(x[1]), reverse=True)[:top_n]
    )
    return sorted_shap


def shap_to_natural_language(shap_dict):
    """
    Convert SHAP dict → readable text.

    Rules:
      • SHAP value is shown as-is (e.g. +0.142 or -0.031)
      • Direction = whether it raises or lowers predicted similarity
      • Strength label based on |SHAP| magnitude
      • NO percentage / contribution language
    """
    if not shap_dict:
        return '      No SHAP explanation available.'

    lines = []
    for feature, shap_val in shap_dict.items():
        direction = '▲ raises similarity' if shap_val > 0 else '▼ lowers similarity'
        mag = abs(shap_val)
        if mag > 0.30:
            strength = 'strong impact'
        elif mag > 0.10:
            strength = 'moderate impact'
        elif mag > 0.03:
            strength = 'small impact'
        else:
            strength = 'negligible'

        feat_display = feature.replace('_', ' ').title()
        lines.append(
            f'      • {feat_display:<28} SHAP={shap_val:+.4f}  {direction}  [{strength}]'
        )
    return '\n'.join(lines)


print('SHAP explainer ready (LinearExplainer, corrected language, FIX 3+5).')


In [ ]:
# ─────────────────────────────────────────────
# SECTION 6C — SHAP Summary Plot
# ─────────────────────────────────────────────
def plot_shap_summary(shap_values, top_n=10, title_suffix=''):
    fnames   = tfidf.get_feature_names_out()
    mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-top_n:][::-1]
    top_feats = [fnames[i] for i in top_idx]
    top_vals  = mean_abs[top_idx]

    plt.figure(figsize=(7, 5))
    plt.barh(range(len(top_feats)), top_vals, color='steelblue')
    plt.yticks(range(len(top_feats)),
               [f.replace('_', ' ').title() for f in top_feats], fontsize=9)
    plt.xlabel('Mean |SHAP value| — average impact on predicted similarity')
    plt.title(f'Top Ingredients Driving Similarity{title_suffix}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('/content/fig14_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig14_shap_summary.png')


In [ ]:
# ─────────────────────────────────────────────
# SECTION 6C — SHAP Summary Plot
# ─────────────────────────────────────────────
def plot_shap_summary(shap_values, top_n=10, title_suffix=''):
    fnames   = tfidf.get_feature_names_out()
    mean_abs = np.abs(shap_values).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[-top_n:][::-1]
    top_feats = [fnames[i] for i in top_idx]
    top_vals  = mean_abs[top_idx]

    plt.figure(figsize=(7, 5))
    plt.barh(range(len(top_feats)), top_vals, color='steelblue')
    plt.yticks(range(len(top_feats)),
               [f.replace('_', ' ').title() for f in top_feats], fontsize=9)
    plt.xlabel('Mean |SHAP value| — average impact on predicted similarity')
    plt.title(f'Top Ingredients Driving Similarity{title_suffix}', fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('/content/fig14_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig14_shap_summary.png')


In [ ]:
# ─────────────────────────────────────────────
# SECTION 7B — Test Cases
# ─────────────────────────────────────────────
print('>>> TEST CASE 1: Comedonal acne')
formula_1 = 'Salicylic Acid + Niacinamide + Zinc PCA'
result1 = skinmeta_recommend(
    acne_type='Comedonal',
    severity=1,
    recommended_formula=formula_1,
    skin_profile=None,
    environmental_factors={'pollution': 'high', 'humidity': 'high', 'uv': 'moderate'},
    plot_shap=True,
)

print('\n\n>>> TEST CASE 2: Inflammatory acne')
formula_2 = 'Benzoyl Peroxide + Niacinamide + Azelaic Acid'
result2 = skinmeta_recommend(
    acne_type='Inflammatory',
    severity=2,
    recommended_formula=formula_2,
    skin_profile=None,
    environmental_factors={'pollution': 'moderate', 'humidity': 'low', 'uv': 'high'},
    plot_shap=True,
)


In [ ]:
# ─────────────────────────────────────────────
# SECTION 8 — Save All Outputs
# ─────────────────────────────────────────────
import pickle

# 1. Tagged product dataset
save_cols = [c for c in [
    'product_name', 'name', brand_col, type_col, 'price_clean',
    ing_col, 'acne_ings_present', 'acne_ingredient_count',
    'acne_types', 'acne_types_str'
] if c and c in acne_products.columns]
acne_products[save_cols].to_csv('acne_products_tagged.csv', index=False)
print('Saved: acne_products_tagged.csv')

# 2. TF-IDF vectorizer
with open('tfidf_product_bridge.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('Saved: tfidf_product_bridge.pkl')

# 3. Sample output JSON
report_result = {
    'acne_type': result1['acne_type'],
    'severity' : result1['severity'],
    'formula'  : result1['formula'],
    'products' : result1['products'],
    }
with open('sample_recommendation_output.json', 'w') as f:
    json.dump(report_result, f, indent=2)
print('Saved: sample_recommendation_output.json')

print('\n' + '=' * 55)
print('  BRIDGE LAYER + XAI SUMMARY')
print('=' * 55)
print(f'  Products indexed : {len(acne_products)}')
print(f'  TF-IDF features  : {len(feature_names)}')
print(f'  XAI method       : SHAP LinearExplainer (Ridge similarity model)')
print(f'  SHAP language    : value + direction only (no % contribution)')
print('=' * 55)
print('  All 5 issues fixed — ready for Phase 3 integration')
print('=' * 55)
